# Step 3 — DistilBERT trust scorer

**Purpose.** Train the primary detector exactly as `eval_protocol.md` §5 fixes it, set its thresholds on validation, and score every validation and test row of `data/passages.jsonl`.

**Models (6 runs):** augmented (headline) and not augmented (ablation) × seeds 42, 43, 44.
**Fixed settings (no tuning, no early stopping; validation only sets thresholds):** `distilbert-base-uncased`, input `[CLS] question [SEP] passage [SEP]`, max 256 tokens with `truncation="only_second"`, AdamW lr 2e-5, weight decay 0.01, batch 16, 3 epochs, linear schedule with 10% warm-up, final-epoch model, class-weighted loss.

**Runtime.** Colab **T4 GPU**. About 4–6 minutes per model, ~30 minutes in total. Each finished model is saved to Drive, so a disconnect resumes where it stopped. Scores are stored by text, so re-running after new rows are added (rewrites, injection) scores only the new pairs.

**Outputs** (Drive, `rag_trust/step3_distilbert/`):
- `scores_distilbert.jsonl` — one line per val/test grouped row: `context_id, passage_id` + one score column per model (`aug_s42`, …). Higher = more likely poisoned.
- `thresholds_distilbert.json` — 5% and 1% FPR thresholds per model, from val n=0 clean passages.
- `train_log.json` — class weights, losses, timings.
- `model_<config>_s<seed>/` — all six models (~1.5 GB). `model_aug_s42` is used for the end-to-end runs; the others let rows added later (rewrites) be scored without retraining.

## 1. Setup

In [ ]:
import os, sys, json, time, random, subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print(gpu or "NO GPU — switch the runtime to T4 GPU (Runtime -> Change runtime type)")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "transformers>=4.45,<5"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/rag_trust"
else:                                   # offline dry run
    DRIVE_ROOT = os.environ["RAG_TRUST_ROOT"]
DATA = f"{DRIVE_ROOT}/data/passages.jsonl"
OUT_DIR = f"{DRIVE_ROOT}/step3_distilbert"
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(DATA), "run notebooks/02_build_dataset.ipynb first"

In [ ]:
CONFIG = {
    "model_name": os.environ.get("DISTILBERT_NAME", "distilbert-base-uncased"),
    "max_length": 256, "truncation": "only_second",
    "lr": 2e-5, "weight_decay": 0.01, "batch_size": 16, "epochs": 3, "warmup_frac": 0.1,
    "seeds": [42, 43, 44], "configs": {"aug": "train_aug", "noaug": "original"},
    "headline": "aug", "fprs": {"fpr5": 0.05, "fpr1": 0.01},
    "score_batch_size": 64,
}
import torch, numpy as np, transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", device)
with open(f"{OUT_DIR}/config.json", "w") as f:
    json.dump({"config": CONFIG, "torch": torch.__version__, "transformers": transformers.__version__,
               "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None}, f, indent=2)

## 2. Shared evaluation code
Identical to `eval_core.py` in the repository. Used by every detector, so all are scored by the same rules.

In [ ]:
%%writefile eval_core.py
"""Shared evaluation for all detectors and baselines (eval_protocol.md section 7).

Every detector writes one score per row of data/passages.jsonl (higher = more likely
poisoned). This module turns rows + scores into the protocol's pairings, rates at frozen
thresholds, bootstrap CIs (resampling questions within each dataset) and the paired
AUROC difference. Pure NumPy, no GPU.
"""
import numpy as np

# dataset groups reported separately (section 4)
GROUPS = {"in_dist": ("nq", "hotpotqa"), "msmarco": ("msmarco",)}


# ---------------------------------------------------------------- metrics

def auroc(pos, neg):
    """AUROC of score(pos) > score(neg), ties counted 0.5 (Mann-Whitney)."""
    pos = np.asarray(pos, float); neg = np.asarray(neg, float)
    if pos.size == 0 or neg.size == 0:
        return float("nan")
    allv = np.concatenate([pos, neg])
    _, inv, counts = np.unique(allv, return_inverse=True, return_counts=True)
    order = allv.argsort(kind="mergesort")
    ranks = np.empty(len(allv)); ranks[order] = np.arange(1, len(allv) + 1)
    ranks = (np.bincount(inv, weights=ranks) / counts)[inv]
    return float((ranks[:pos.size].sum() - pos.size * (pos.size + 1) / 2) / (pos.size * neg.size))


def threshold_at_fpr(neg_scores, fpr):
    """Threshold t such that flagging score > t gives FPR <= fpr on these negatives
    (section 7: set on val n=0 clean passages, then frozen)."""
    s = np.sort(np.asarray(neg_scores, float))[::-1]
    k = int(np.floor(fpr * len(s)))          # number of false positives allowed
    return float(s[k]) if k < len(s) else float(s[-1]) - 1e-12


def rate(scores, t):
    """Share of scores flagged (score > t): TPR on positives, FPR on negatives."""
    s = np.asarray(scores, float)
    return float((s > t).mean()) if s.size else float("nan")


# ---------------------------------------------------------------- pairings (section 7)

def _is(r, **kw):
    return all(r.get(k) == v for k, v in kw.items())


def pairing(rows, name):
    """Return (positive_rows, negative_rows) for one protocol pairing.
    Positives at n=1 use the five n=1 detection groups (headline condition)."""
    orig_pos = [r for r in rows if _is(r, group_variant="original", n_poison=1, label=1)]
    orig_neg = [r for r in rows if _is(r, group_variant="original", n_poison=0, label=0)]
    if name == "original":
        return orig_pos, orig_neg
    if name == "original_n5":
        return [r for r in rows if _is(r, group_variant="original", n_poison=5, label=1)], orig_neg
    if name == "prefix_poison_side":
        return [r for r in rows if _is(r, group_variant="prefix_removed", n_poison=1, variant="prefix_removed")], orig_neg
    if name == "prefix_clean_side":
        return orig_pos, [r for r in rows if _is(r, group_variant="clean_prefixed", variant="clean_prefixed")]
    if name == "llm_style":
        return orig_pos, [r for r in rows if _is(r, group_variant="rewritten_clean", variant="rewritten_clean")]
    if name == "shortcut_controlled":
        return ([r for r in rows if _is(r, group_variant="shortcut_controlled", n_poison=1, variant="prefix_removed")],
                [r for r in rows if _is(r, group_variant="shortcut_controlled", n_poison=0, variant="clean_truncated")])
    if name == "unseen_attack":
        pos = [r for r in rows if _is(r, group_variant="injection", variant="injection")]
        qs = {(r["dataset"], r["qid"]) for r in pos}
        return pos, [r for r in orig_neg if (r["dataset"], r["qid"]) in qs]
    if name == "collateral_n1":       # FPR only: clean passages inside attacked n=1 groups
        return [], [r for r in rows if _is(r, group_variant="original", n_poison=1, label=0)]
    raise ValueError(name)


PAIRINGS = ["original", "original_n5", "prefix_poison_side", "prefix_clean_side", "llm_style",
            "shortcut_controlled", "unseen_attack", "collateral_n1"]


# ---------------------------------------------------------------- bootstrap by question

def _by_question(rs, score_key):
    out = {}
    for r in rs:
        out.setdefault((r["dataset"], r["qid"]), []).append(r[score_key])
    return out


def _resample_keys(keys, rng):
    """Resample questions within each dataset (so one dataset cannot dominate)."""
    by_ds = {}
    for k in keys:
        by_ds.setdefault(k[0], []).append(k)
    out = []
    for ds in sorted(by_ds):
        ks = by_ds[ds]
        out += [ks[i] for i in rng.integers(0, len(ks), len(ks))]
    return out


def evaluate(rows, score_key, thresholds, n_boot=2000, seed=42, pairings=PAIRINGS, groups=GROUPS):
    """Detection metrics for one score column. thresholds: {"fpr5": t5, "fpr1": t1}.
    Returns {group: {pairing: {...}}} with point estimates, bootstrap CIs and counts."""
    res = {}
    test = [r for r in rows if r["split"] == "test"]
    for g, dsets in groups.items():
        sub = [r for r in test if r["dataset"] in dsets]
        res[g] = {}
        for name in pairings:
            pos, neg = pairing(sub, name)
            if not pos and not neg:
                continue
            P, N = _by_question(pos, score_key), _by_question(neg, score_key)
            keys = sorted(set(P) | set(N))
            ps = [s for k in keys for s in P.get(k, [])]
            ns = [s for k in keys for s in N.get(k, [])]
            out = {"n_pos": len(ps), "n_neg": len(ns), "n_questions": len(keys)}
            point = {"auroc": auroc(ps, ns)}
            for tk, t in thresholds.items():
                point[f"tpr_{tk}"] = rate(ps, t); point[f"fpr_{tk}"] = rate(ns, t)
            rng = np.random.default_rng(seed); boots = {k: [] for k in point}
            for _ in range(n_boot):
                ks = _resample_keys(keys, rng)
                bp = [s for k in ks for s in P.get(k, [])]; bn = [s for k in ks for s in N.get(k, [])]
                boots["auroc"].append(auroc(bp, bn))
                for tk, t in thresholds.items():
                    boots[f"tpr_{tk}"].append(rate(bp, t)); boots[f"fpr_{tk}"].append(rate(bn, t))
            for k, v in point.items():
                b = np.asarray(boots[k], float); b = b[~np.isnan(b)]
                out[k] = {"value": None if np.isnan(v) else round(v, 4),
                          "ci95": [round(float(x), 4) for x in np.quantile(b, [0.025, 0.975])] if b.size else None}
            res[g][name] = out
    return res


def paired_auroc_difference(rows, key_a, key_b, name, group_dsets, n_boot=2000, seed=42):
    """Section 7: bootstrap the difference AUROC(a) - AUROC(b) on the same rows,
    resampling questions within dataset. Returns value, 95% CI and whether it excludes 0."""
    test = [r for r in rows if r["split"] == "test" and r["dataset"] in group_dsets]
    pos, neg = pairing(test, name)
    keys = sorted({(r["dataset"], r["qid"]) for r in pos + neg})
    Pa, Na, Pb, Nb = (_by_question(x, k) for x, k in ((pos, key_a), (neg, key_a), (pos, key_b), (neg, key_b)))
    def diff(ks):
        a = auroc([s for k in ks for s in Pa.get(k, [])], [s for k in ks for s in Na.get(k, [])])
        b = auroc([s for k in ks for s in Pb.get(k, [])], [s for k in ks for s in Nb.get(k, [])])
        return a - b
    point = diff(keys); rng = np.random.default_rng(seed)
    boots = np.array([diff(_resample_keys(keys, rng)) for _ in range(n_boot)])
    boots = boots[~np.isnan(boots)]
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return {"diff": round(float(point), 4), "ci95": [round(float(lo), 4), round(float(hi), 4)],
            "excludes_zero": bool(lo > 0 or hi < 0)}


def val_clean_n0(rows, score_key):
    """Section 7: thresholds are set on val n=0 clean passages (NQ + HotpotQA val)."""
    return [r[score_key] for r in rows
            if r["split"] == "val" and _is(r, group_variant="original", n_poison=0, label=0)]

In [ ]:
import importlib, eval_core as ec
importlib.reload(ec)
ROWS = [json.loads(l) for l in open(DATA)]
print(f"{len(ROWS):,} rows")

## 3. Training sets (protocol §5)
Unique passages of the train questions, de-duplicated by `passage_id`: 5 poison + clean top-5 + ranks 6–20 per question. Same draws as the feature scorer.

In [ ]:
def train_set(rows, group_variant):
    seen = {}
    for r in rows:
        if r["split"] == "train" and r["group_variant"] == group_variant:
            seen.setdefault(r["passage_id"], (r["question"], r["passage"], r["label"]))
    return [seen[k] for k in sorted(seen)]          # fixed order -> reproducible

TRAIN = {name: train_set(ROWS, gv) for name, gv in CONFIG["configs"].items()}
for name, data in TRAIN.items():
    n1 = sum(l for _, _, l in data)
    print(f"{name:6} {len(data)} passages: {n1} poisoned, {len(data) - n1} clean")

## 4. Training and scoring

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)

def collate(batch, tok):
    enc = tok.pad([{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch],
                  return_tensors="pt")
    if "label" in batch[0]:
        enc["labels"] = torch.tensor([b["label"] for b in batch])
    return enc

def train_model(data, seed):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(CONFIG["model_name"])
    model = AutoModelForSequenceClassification.from_pretrained(CONFIG["model_name"], num_labels=2).to(device)
    enc = tok([q for q, _, _ in data], [p for _, p, _ in data], truncation=CONFIG["truncation"],
              max_length=CONFIG["max_length"])
    items = [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i], "label": data[i][2]}
             for i in range(len(data))]
    n, n1 = len(data), sum(d[2] for d in data)
    weights = torch.tensor([n / (2 * (n - n1)), n / (2 * n1)], dtype=torch.float, device=device)   # inverse frequency
    gen = torch.Generator().manual_seed(seed)
    loader = torch.utils.data.DataLoader(items, batch_size=CONFIG["batch_size"], shuffle=True, generator=gen,
                                         collate_fn=lambda b: collate(b, tok))
    no_decay = ["bias", "LayerNorm.weight"]                # standard practice: no decay on biases / LayerNorm
    groups = [{"params": [p for nm, p in model.named_parameters() if not any(x in nm for x in no_decay)],
               "weight_decay": CONFIG["weight_decay"]},
              {"params": [p for nm, p in model.named_parameters() if any(x in nm for x in no_decay)],
               "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=CONFIG["lr"])
    total = len(loader) * CONFIG["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(CONFIG["warmup_frac"] * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    log = {"class_weights": weights.tolist(), "n_train": n, "n_poison": n1, "epoch_loss": []}
    model.train()
    for ep in range(CONFIG["epochs"]):
        tot = 0.0
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(labels)
        log["epoch_loss"].append(round(tot / n, 4))
        print(f"    epoch {ep + 1}: mean loss {tot / n:.4f}")
    return model, tok, log

@torch.no_grad()
def score_texts(model, tok, pairs):
    """P(poisoned) for each (question, passage) pair."""
    model.eval()
    order = sorted(range(len(pairs)), key=lambda i: len(pairs[i][1]))      # length-sorted batches: faster
    out = np.zeros(len(pairs))
    bs = CONFIG["score_batch_size"]
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok([pairs[i][0] for i in idx], [pairs[i][1] for i in idx], truncation=CONFIG["truncation"],
                  max_length=CONFIG["max_length"], padding=True, return_tensors="pt").to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(**enc).logits.float()
        out[idx] = torch.softmax(logits, -1)[:, 1].cpu().numpy()
    return out

# every val/test row in a group gets a score; DistilBERT ignores the group, so score each distinct text once
EVAL_ROWS = [r for r in ROWS if r["split"] in ("val", "test") and r["context_id"] is not None]
import hashlib
def pair_key(q, p):
    # scores are stored by text, not by position, so rows added later (rewrites) can't be misaligned
    return hashlib.sha1((q + "" + p).encode()).hexdigest()
PAIRS = {pair_key(r["question"], r["passage"]): (r["question"], r["passage"]) for r in EVAL_ROWS}
print(f"{len(EVAL_ROWS):,} val/test rows; {len(PAIRS):,} distinct (question, passage) pairs to score")

In [ ]:
# Train all 6 models (each saved to Drive), then score every pair not scored yet.
# Re-running skips finished models and only scores new pairs (e.g. after rewritten_clean is added).
RUN_LOG = f"{OUT_DIR}/train_log.json"
train_log = json.load(open(RUN_LOG)) if os.path.exists(RUN_LOG) else {}
for name in CONFIG["configs"]:
    for seed in CONFIG["seeds"]:
        key = f"{name}_s{seed}"; mdir = f"{OUT_DIR}/model_{key}"; spath = f"{OUT_DIR}/pair_scores_{key}.json"
        scores = json.load(open(spath)) if os.path.exists(spath) else {}
        if os.path.exists(f"{mdir}/config.json"):
            missing = [k for k in PAIRS if k not in scores]
            if not missing:
                print(f"{key}: trained and fully scored"); continue
            print(f"{key}: scoring {len(missing)} new pairs with the saved model")
            tok = AutoTokenizer.from_pretrained(mdir)
            model = AutoModelForSequenceClassification.from_pretrained(mdir).to(device)
        else:
            print(f"{key}: training on {len(TRAIN[name])} passages")
            t0 = time.time()
            model, tok, log = train_model(TRAIN[name], seed)
            log["train_seconds"] = round(time.time() - t0, 1)
            model.save_pretrained(mdir); tok.save_pretrained(mdir)
            train_log[key] = log
            missing = list(PAIRS)
        t1 = time.time()
        new = score_texts(model, tok, [PAIRS[k] for k in missing])
        scores.update({k: float(v) for k, v in zip(missing, new)})
        json.dump(scores, open(spath, "w"))
        secs = time.time() - t1
        train_log.setdefault(key, {})["score_ms_per_passage"] = round(1000 * secs / max(len(missing), 1), 3)
        json.dump(train_log, open(RUN_LOG, "w"), indent=2)
        print(f"    scored {len(missing)} pairs in {secs:.1f}s")
        del model; torch.cuda.empty_cache() if device == "cuda" else None

## 5. Thresholds (val n=0 clean passages) and the score file

In [ ]:
MODELS = [f"{n}_s{s}" for n in CONFIG["configs"] for s in CONFIG["seeds"]]
PAIR_SCORES = {m: json.load(open(f"{OUT_DIR}/pair_scores_{m}.json")) for m in MODELS}
for r in EVAL_ROWS:
    k = pair_key(r["question"], r["passage"])
    for m in MODELS:
        r[m] = PAIR_SCORES[m][k]

THRESHOLDS = {}
for m in MODELS:
    neg = ec.val_clean_n0(EVAL_ROWS, m)
    THRESHOLDS[m] = {k: {"threshold": ec.threshold_at_fpr(neg, f),
                         "val_fpr_achieved": ec.rate(neg, ec.threshold_at_fpr(neg, f)), "n_val_clean": len(neg)}
                     for k, f in CONFIG["fprs"].items()}
TH_PATH = f"{OUT_DIR}/thresholds_distilbert.json"
if os.path.exists(TH_PATH):                     # frozen: a re-run must reproduce them exactly
    old = json.load(open(TH_PATH))["thresholds"]
    assert all(abs(old[m][k]["threshold"] - THRESHOLDS[m][k]["threshold"]) < 1e-9
               for m in MODELS for k in CONFIG["fprs"]), "thresholds changed: validation data must not change"
else:
    json.dump({"thresholds": THRESHOLDS, "note": "set on val n=0 clean passages (NQ+HotpotQA); frozen for every test set"},
              open(TH_PATH, "w"), indent=2)

with open(f"{OUT_DIR}/scores_distilbert.jsonl", "w") as f:
    for r in EVAL_ROWS:
        f.write(json.dumps({"context_id": r["context_id"], "passage_id": r["passage_id"],
                            **{m: round(r[m], 6) for m in MODELS}}) + "\n")
print({m: {k: round(v["threshold"], 4) for k, v in t.items()} for m, t in THRESHOLDS.items()})

## 6. Results preview
Seed 42 gets bootstrap CIs (protocol §7); the table shows mean ± sd across the 3 seeds. The final evaluation notebook combines these scores with the feature scorer and baselines; this is a first look.

In [ ]:
import pandas as pd
th = lambda m: {k: v["threshold"] for k, v in THRESHOLDS[m].items()}
PREVIEW = ["original", "prefix_poison_side", "prefix_clean_side", "llm_style", "shortcut_controlled", "unseen_attack", "collateral_n1"]
res = {m: ec.evaluate(EVAL_ROWS, m, th(m), n_boot=(2000 if m.endswith("_s42") else 0), pairings=PREVIEW)
       for m in MODELS}
json.dump(res, open(f"{OUT_DIR}/detection_preview.json", "w"), indent=2)

def cell(name, g, p, metric):
    vals = [res[f"{name}_s{s}"][g].get(p, {}).get(metric, {}).get("value") for s in CONFIG["seeds"]]
    vals = [v for v in vals if v is not None]
    return f"{np.mean(vals):.3f} ± {np.std(vals):.3f}" if vals else "—"
for metric in ("auroc", "tpr_fpr5", "fpr_fpr5"):
    print(f"\n{metric} (mean ± sd over seeds 42/43/44)")
    display(pd.DataFrame({f"{n} | {g}": {p: cell(n, g, p, metric) for p in PREVIEW}
                          for n in CONFIG["configs"] for g in ec.GROUPS}))
s42 = res[f"{CONFIG['headline']}_s42"]
print("\nHeadline model (aug, seed 42) with 95% bootstrap CIs:")
for g, v in s42.items():
    for p, mm in v.items():
        a = mm["auroc"]
        print(f"  {g:8} {p:20} AUROC {a['value']} {a['ci95']}  TPR@5% {mm['tpr_fpr5']['value']}  "
              f"FPR@5% {mm['fpr_fpr5']['value']}  (pos {mm['n_pos']}, neg {mm['n_neg']})")

## 7. What to check, then commit

1. **Training ran on 3,000 passages per model** (600 poisoned) and the loss falls across the 3 epochs.
2. **Validation FPR achieved** is 0.05 / 0.01 (or just under, from ties).
3. **Read the preview, don't tune.** If Original is near 1.0 and the shortcut-controlled pairing drops, that is the expected shortcut finding; if the augmented model holds up better than the ablation, that is the augmentation result.
4. **Commit to GitHub:** this notebook, `eval_core.py`, and from Drive `step3_distilbert/`: `scores_distilbert.jsonl` (gzip it if it is over 25 MB), `thresholds_distilbert.json`, `train_log.json`, `config.json`, `detection_preview.json`. Keep the model folders on Drive.